# 02. Результаты эксперимента

Читает артефакты опубликованного full run (`reports/runs/<run_id>/`). Пересчёт: `hmrp run --config configs/full.yaml`.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 160, "display.max_columns", 30)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
REP = ROOT / "reports"
def read(p): return json.loads(Path(p).read_text(encoding="utf-8"))

In [ ]:
idx = read(REP / "runs" / "index.json"); RUN = REP / "runs" / idx["full"]
dec = read(RUN / "selection_decision.json"); test = read(RUN / "test_metrics.json")
cal = read(RUN / "calibration.json"); boot = read(RUN / "bootstrap.json")
idx["full"], dec["selected_model"], dec["near_best"]

## Development walk-forward

In [ ]:
dev = pd.read_csv(RUN / "dev_results.csv")
v = dev[dev.split == "validation"]
v.pivot_table(index=["model", "config"], columns="fold", values="average_precision").assign(mean=lambda d: d.mean(axis=1)).sort_values("mean", ascending=False).round(5)

In [ ]:
pd.DataFrame(dec["development_scores"]).T

## Калибровка

In [ ]:
pd.DataFrame({m: r["params"] for m, r in cal["models"].items()}).T

## Final test

In [ ]:
rows = []
for m, r in test["models"].items():
    rows.append({"model": m, "selected": r["selected"], "N": r["n"], "prevalence": r["prevalence"], "AP": r["average_precision"],
                 "ROC_AUC": r["roc_auc"], "P@5%": r["at_budget"]["top_5pct"]["precision"], "Lift@5%": r["at_budget"]["top_5pct"]["lift"],
                 "Brier_raw": r["raw_probability"].get("brier"), "Brier_cal": r["calibrated_probability"].get("brier")})
pd.DataFrame(rows).round(5)

In [ ]:
pd.DataFrame({k: {m: f"[{v['low']:.4f}; {v['high']:.4f}]" for m, v in d.items()} for k, d in boot["differences"].items()})

In [ ]:
curves = pd.read_csv(RUN / "test_budget_curves.csv")
ax = curves.pivot(index="budget", columns="model", values="precision").plot(figsize=(8, 3.5), linewidth=2, title="Final test: Precision@top-k")
ax.set_xlabel("share of audience"); ax.set_ylabel("precision")

In [ ]:
rel = pd.read_csv(RUN / "test_reliability_bins.csv")
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, kind in zip(axes, ["raw", "calibrated"]):
    for m, g in rel[rel.kind == kind].groupby("model"):
        ax.plot(g.mean_pred, g.frac_pos, marker="o", label=m)
    ax.plot([0, 1], [0, 1], color="grey", linewidth=1); ax.set_title(kind); ax.legend(fontsize=8)

## Batch scoring

In [ ]:
pd.DataFrame([read(p) for p in sorted((RUN / "scoring").glob("*.json"))])[["as_of_date", "model_version", "n_scored", "k", "primary_probability_field", "calibration_fallback_used", "label_window_observed"]]